In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path("..")                   
DATA_DIR = BASE_DIR / "data" / "data_raw"

COLL_PATH = DATA_DIR / "Traffic_Collisions 2023-2025.csv"
WEATHER_PATH = DATA_DIR / "weather raw 23-25.xlsx"
OUT_DIR = BASE_DIR / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("COLL_PATH:", COLL_PATH, "exists?", COLL_PATH.exists())
print("WEATHER_PATH:", WEATHER_PATH, "exists?", WEATHER_PATH.exists())
print("OUT_DIR:", OUT_DIR)

COLL_PATH: ..\data\data_raw\Traffic_Collisions 2023-2025.csv exists? True
WEATHER_PATH: ..\data\data_raw\weather raw 23-25.xlsx exists? True
OUT_DIR: ..\data\processed


In [7]:
# =========================================================
# COLLISIONS (EVENT LEVEL): LOAD + CLEAN
# - Delete NSA
# - Delete N/R or NR rows for selected columns
# - Fix date/hour and build date_time_hour
# - Build HOOD_158_CODE
# - Fix coordinates (keep LAT_WGS84/LONG_WGS84, drop X/Y)
# - Encode YES/NO => 0/1 integers (uint8), no decimals
# =========================================================

coll = pd.read_csv(COLL_PATH, low_memory=False)
coll.columns = [str(c).strip() for c in coll.columns]

# --- 1) Delete NSA HOOD rows ---
hood_upper = coll["HOOD_158"].astype(str).str.strip().str.upper()
coll = coll[hood_upper.ne("NSA")].copy()

# --- 2) Delete N/R or NR rows in important columns ---
# (You requested to delete them. Add/remove columns here if needed.)
nr_cols = [
    "DIVISION",
    "INJURY_COLLISIONS", "FTR_COLLISIONS", "PD_COLLISIONS",
    "PEDESTRIAN", "BICYCLE", "AUTOMOBILE", "MOTORCYCLE", "PASSENGER"
]
nr_cols = [c for c in nr_cols if c in coll.columns]

for c in nr_cols:
    x = coll[c].astype(str).str.strip().str.upper()
    coll = coll[~x.isin(["N/R", "NR"])].copy()

# --- 3) Time: keep OCC_DATE and OCC_HOUR, build date_time_hour ---
coll["OCC_DATE"] = pd.to_datetime(coll["OCC_DATE"], errors="coerce").dt.floor("D")

coll["OCC_HOUR"] = pd.to_numeric(coll["OCC_HOUR"], errors="coerce")
coll.loc[~coll["OCC_HOUR"].between(0, 23), "OCC_HOUR"] = np.nan
coll["OCC_HOUR"] = coll["OCC_HOUR"].astype("Int64")

coll["date_time_hour"] = coll["OCC_DATE"] + pd.to_timedelta(
    coll["OCC_HOUR"].fillna(0).astype(int), unit="h"
)

# --- 4) HOOD code: make 3-digit string ---
hood_digits = coll["HOOD_158"].astype(str).str.extract(r"(\d+)", expand=False)
hood_num = pd.to_numeric(hood_digits, errors="coerce").astype("Int64")
coll["HOOD_158_CODE"] = hood_num.astype(str).str.zfill(3).replace("<NA>", np.nan)
coll = coll[coll["HOOD_158_CODE"].notna()].copy()

# --- 5) Coordinates: keep LAT_WGS84/LONG_WGS84; if X/Y exist, drop them ---
for c in ["LAT_WGS84", "LONG_WGS84", "X", "Y", "x", "y"]:
    if c in coll.columns:
        coll[c] = pd.to_numeric(coll[c], errors="coerce")

# If LAT/LONG are missing but X/Y exist, use Y as lat and X as lon
if ("LAT_WGS84" not in coll.columns or "LONG_WGS84" not in coll.columns) and ("Y" in coll.columns and "X" in coll.columns):
    coll["LAT_WGS84"] = coll["Y"]
    coll["LONG_WGS84"] = coll["X"]
if ("LAT_WGS84" not in coll.columns or "LONG_WGS84" not in coll.columns) and ("y" in coll.columns and "x" in coll.columns):
    coll["LAT_WGS84"] = coll["y"]
    coll["LONG_WGS84"] = coll["x"]

# Drop X/Y duplicates
coll = coll.drop(columns=[c for c in ["X","Y","x","y"] if c in coll.columns], errors="ignore")

# Remove invalid (0,0) coords
if "LAT_WGS84" in coll.columns and "LONG_WGS84" in coll.columns:
    coll = coll[~((coll["LAT_WGS84"].fillna(0) == 0) & (coll["LONG_WGS84"].fillna(0) == 0))].copy()

# --- 6) Encode YES/NO -> strict 0/1 integer (uint8), NO decimals ---
yes_no_cols = [
    "INJURY_COLLISIONS","FTR_COLLISIONS","PD_COLLISIONS",
    "PEDESTRIAN","BICYCLE","AUTOMOBILE","MOTORCYCLE","PASSENGER"
]
yes_no_cols = [c for c in yes_no_cols if c in coll.columns]

for c in yes_no_cols:
    x = coll[c].astype(str).str.strip().str.upper()
    coll[c + "_01"] = (x == "YES").astype("uint8")   # always integer 0/1

# --- 7) Deduplicate by EVENT_UNIQUE_ID (safety) ---
if "EVENT_UNIQUE_ID" in coll.columns:
    coll = coll.sort_values("date_time_hour").drop_duplicates("EVENT_UNIQUE_ID", keep="first").copy()

# --- 8) Keep only event-level columns needed for dashboard/drilldown ---
keep_event = [
    "EVENT_UNIQUE_ID",
    "OCC_DATE","OCC_HOUR","date_time_hour",
    "HOOD_158_CODE","NEIGHBOURHOOD_158",
    "DIVISION",
    "LAT_WGS84","LONG_WGS84",
    "FATALITIES",
    "INJURY_COLLISIONS_01","FTR_COLLISIONS_01","PD_COLLISIONS_01",
    "PEDESTRIAN_01","BICYCLE_01","AUTOMOBILE_01","MOTORCYCLE_01","PASSENGER_01",
]
keep_event = [c for c in keep_event if c in coll.columns]
coll_event_clean = coll[keep_event].copy()

# Save event-level cleaned dataset (dashboard drilldown)
coll_event_clean.to_csv(OUT_DIR / "collisions_clean_event.csv", index=False)

In [8]:
# =========================================================
# COLLISIONS: AGGREGATE TO HOOD × 3-HOUR BLOCK
# - Adds time features: block_hour, OCC_DOW, OCC_MONTH, is_weekend
# - Keeps type/user counts (injury/ftr/pd/ped/bike etc.)
# - (NO year)
# =========================================================

coll_event_clean["time_3h"] = pd.to_datetime(coll_event_clean["date_time_hour"], errors="coerce").dt.floor("3h")

coll_3h = (coll_event_clean.groupby(["HOOD_158_CODE","time_3h"], as_index=False)
           .agg(
               collisions=("EVENT_UNIQUE_ID","count") if "EVENT_UNIQUE_ID" in coll_event_clean.columns else ("time_3h","size"),
               injury_collisions=("INJURY_COLLISIONS_01","sum") if "INJURY_COLLISIONS_01" in coll_event_clean.columns else ("time_3h","size"),
               ftr_collisions=("FTR_COLLISIONS_01","sum") if "FTR_COLLISIONS_01" in coll_event_clean.columns else ("time_3h","size"),
               pd_collisions=("PD_COLLISIONS_01","sum") if "PD_COLLISIONS_01" in coll_event_clean.columns else ("time_3h","size"),
               fatalities=("FATALITIES","sum") if "FATALITIES" in coll_event_clean.columns else ("time_3h","size"),

               pedestrian_collisions=("PEDESTRIAN_01","sum") if "PEDESTRIAN_01" in coll_event_clean.columns else ("time_3h","size"),
               bicycle_collisions=("BICYCLE_01","sum") if "BICYCLE_01" in coll_event_clean.columns else ("time_3h","size"),
               automobile_collisions=("AUTOMOBILE_01","sum") if "AUTOMOBILE_01" in coll_event_clean.columns else ("time_3h","size"),
               motorcycle_collisions=("MOTORCYCLE_01","sum") if "MOTORCYCLE_01" in coll_event_clean.columns else ("time_3h","size"),
               passenger_collisions=("PASSENGER_01","sum") if "PASSENGER_01" in coll_event_clean.columns else ("time_3h","size"),
           ))

# Force integer counts (no decimals)
for c in coll_3h.columns:
    if c not in ["HOOD_158_CODE","time_3h"]:
        coll_3h[c] = pd.to_numeric(coll_3h[c], errors="coerce").fillna(0).astype(int)

# Time features (important for model)
coll_3h["block_hour"] = coll_3h["time_3h"].dt.hour                 # 0,3,6,...,21
coll_3h["OCC_DOW"] = coll_3h["time_3h"].dt.day_name()              # names for dashboard
coll_3h["OCC_MONTH"] = coll_3h["time_3h"].dt.month_name()          # names for dashboard
coll_3h["dow_num"] = coll_3h["time_3h"].dt.dayofweek               # 0-6 numeric for modelling
coll_3h["month_num"] = coll_3h["time_3h"].dt.month                 # 1-12 numeric for modelling
coll_3h["is_weekend"] = (coll_3h["dow_num"] >= 5).astype(int)

# (Optional) save aggregated collisions
coll_3h.to_csv(OUT_DIR / "collisions_3h_hood.csv", index=False)

In [9]:
# =========================================================
# WEATHER (HOURLY): LOAD + CLEAN (DST safe)
# - Drops blank rows & unnamed columns
# - Creates date_time_hour from DMY + Time
# - Fixes DST: collapse duplicates + fill missing hours + interpolate
# =========================================================

weather = pd.read_excel(WEATHER_PATH, engine="openpyxl")
weather.columns = [str(c).strip() for c in weather.columns]

# Drop blank rows + unnamed columns
weather = weather[weather["DMY"].notna() & weather["Time"].notna()].copy()
weather = weather.drop(columns=[c for c in weather.columns if str(c).lower().startswith("unnamed")], errors="ignore")

# Build hourly datetime
weather["DMY"] = pd.to_datetime(weather["DMY"], errors="coerce").dt.floor("D")
hour_str = weather["Time"].astype(str).str.strip().str.extract(r"(\d{1,2})", expand=False)
weather["hour"] = pd.to_numeric(hour_str, errors="coerce")
weather.loc[~weather["hour"].between(0, 23), "hour"] = np.nan
weather = weather[weather["DMY"].notna() & weather["hour"].notna()].copy()

weather["date_time_hour"] = weather["DMY"] + pd.to_timedelta(weather["hour"].astype(int), unit="h")

# Numeric columns (only convert those that exist)
num_weather_cols = [
    "pressure_sea","wind_speed","relative_humidity","temperature",
    "visibility","cloud_cover_8","rain","snow","snow_on_ground"
]
for c in num_weather_cols:
    if c in weather.columns:
        weather[c] = pd.to_numeric(weather[c], errors="coerce")

# DST fix: collapse duplicate hours (fall-back)
if weather["date_time_hour"].duplicated().any():
    agg = {c: "mean" for c in num_weather_cols if c in weather.columns}
    weather = weather.groupby("date_time_hour", as_index=False).agg(agg)

# DST fix: fill missing hours (spring-forward) by reindexing full hourly range
full_hours = pd.date_range(weather["date_time_hour"].min(), weather["date_time_hour"].max(), freq="h")
weather = (weather.set_index("date_time_hour")
                 .reindex(full_hours)
                 .rename_axis("date_time_hour")
                 .reset_index())

# Interpolate numeric gaps (time-based)
weather = weather.set_index("date_time_hour")
for c in num_weather_cols:
    if c in weather.columns:
        weather[c] = weather[c].interpolate(method="time", limit_direction="both")
weather = weather.reset_index()

weather_hourly_clean = weather[["date_time_hour"] + [c for c in num_weather_cols if c in weather.columns]].copy()

# Save hourly clean weather (optional)
weather_hourly_clean.to_csv(OUT_DIR / "weather_clean_hourly.csv", index=False, float_format="%.3f")

In [10]:
# =========================================================
# WEATHER: AGGREGATE TO 3-HOUR BLOCK
# - Mean: temperature, wind_speed, humidity, pressure, cloud_cover
# - Sum: rain, snow
# - Min: visibility (worst visibility)
# - Max: snow_on_ground
# - Round floats to 3 decimals for clean CSV
# =========================================================

weather_hourly_clean["time_3h"] = pd.to_datetime(weather_hourly_clean["date_time_hour"], errors="coerce").dt.floor("3h")

agg_weather = {}

# Mean (conditions)
for c in ["pressure_sea","wind_speed","relative_humidity","temperature","cloud_cover_8"]:
    if c in weather_hourly_clean.columns:
        agg_weather[c] = "mean"

# Sum (amount over 3h)
for c in ["rain","snow"]:
    if c in weather_hourly_clean.columns:
        agg_weather[c] = "sum"

# Worst visibility
if "visibility" in weather_hourly_clean.columns:
    agg_weather["visibility"] = "min"

# State variable
if "snow_on_ground" in weather_hourly_clean.columns:
    agg_weather["snow_on_ground"] = "max"

weather_3h = weather_hourly_clean.groupby("time_3h", as_index=False).agg(agg_weather)

# Round floats to 3 decimals
float_cols = weather_3h.select_dtypes(include=["float"]).columns
weather_3h[float_cols] = weather_3h[float_cols].round(3)

weather_3h.to_csv(OUT_DIR / "weather_3h.csv", index=False, float_format="%.3f")

In [11]:
# =========================================================
# MERGE + BUILD MODEL DATASET
# - Merge on time_3h
# - Keep important features for modelling (compact)
# - Drop rate columns (we are NOT creating rates here)
# - Keep day/month numeric (dow_num, month_num) for modelling
# =========================================================

merged_3h = coll_3h.merge(weather_3h, on="time_3h", how="left", validate="m:1")

# ---- Model feature set (compact but rich) ----
# Keys:
#   HOOD_158_CODE, time_3h
# Collision history counts:
#   collisions + injury/ftr/pd + pedestrian/bicycle (good for insight and modelling)
# Time:
#   block_hour, dow_num, month_num, is_weekend
# Weather (strong predictors):
#   temperature, visibility, rain, snow, snow_on_ground, wind_speed, relative_humidity, pressure_sea, cloud_cover_8 (if present)

keep_model = [
    "HOOD_158_CODE","time_3h",
    "collisions","injury_collisions","ftr_collisions","pd_collisions",
    "pedestrian_collisions","bicycle_collisions",
    "block_hour","dow_num","month_num","is_weekend",
    "temperature","visibility","rain","snow","snow_on_ground",
    "wind_speed","relative_humidity","pressure_sea","cloud_cover_8"
]
keep_model = [c for c in keep_model if c in merged_3h.columns]
model_df = merged_3h[keep_model].copy()

# Round float columns for clean display
float_cols = model_df.select_dtypes(include=["float"]).columns
model_df[float_cols] = model_df[float_cols].round(3)

# Save final modelling dataset
model_df.to_csv(OUT_DIR / "model_hood_3h_weather.csv", index=False, float_format="%.3f")

# (Optional) Save merged dataset for dashboard analysis too (includes DOW/MONTH names)
merged_3h.to_csv(OUT_DIR / "dashboard_hood_3h_weather.csv", index=False, float_format="%.3f")